In [0]:
WITH who AS (
  SELECT SpatialDim AS iso3, TimeDim AS year, NumericValue AS who_value
  FROM workspace.public_health.bronze_who
  WHERE IndicatorCode = 'WHOSIS_000001' AND Dim1 = 'SEX_BTSX'
),
wb AS (
  SELECT countryiso3code AS iso3, CAST(date AS INT) AS year, value AS wb_value
  FROM workspace.public_health.bronze_worldbank
  WHERE indicator.id = 'SP.DYN.LE00.IN' AND value IS NOT NULL
),
owid AS (
  SELECT code AS iso3, year, life_expectancy_0 AS owid_value
  FROM workspace.public_health.bronze_owid_life_expectancy
  WHERE code IS NOT NULL AND code NOT LIKE 'OWID%'
)
SELECT
  who.iso3, who.year,
  ROUND(who_value, 1) AS who,
  ROUND(wb_value, 1) AS world_bank,
  ROUND(owid_value, 1) AS owid,
  ROUND(GREATEST(who_value, wb_value, owid_value) - LEAST(who_value, wb_value, owid_value), 1) AS spread
FROM who
JOIN wb ON who.iso3 = wb.iso3 AND who.year = wb.year
JOIN owid ON who.iso3 = owid.iso3 AND who.year = owid.year
WHERE who.year = 2019
ORDER BY spread DESC
LIMIT 15;

In [0]:
WITH who AS (
  SELECT TimeDim AS year, NumericValue AS who_value
  FROM workspace.public_health.bronze_who
  WHERE IndicatorCode = 'WHOSIS_000001' AND Dim1 = 'SEX_BTSX' AND SpatialDim = 'CAF'
),
wb AS (
  SELECT CAST(date AS INT) AS year, value AS wb_value
  FROM workspace.public_health.bronze_worldbank
  WHERE indicator.id = 'SP.DYN.LE00.IN' AND countryiso3code = 'CAF'
)
SELECT wb.year, ROUND(who_value, 1) AS who, ROUND(wb_value, 1) AS world_bank
FROM wb LEFT JOIN who ON wb.year = who.year
WHERE wb.year BETWEEN 2010 AND 2023
ORDER BY wb.year;